# People Analytics — Preparação e Tratamento dos Dados (ETL)

Este notebook importa a base bruta (`data/raw`), diagnostica problemas de qualidade, aplica os tratamentos necessários, valida o resultado e exporta a base tratada para `data/processed`.

## 1. Preparação do Ambiente

In [1]:
# Importa as bibliotecas utilizadas na preparação dos dados
from pathlib import Path

import pandas as pd
import numpy as np

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Pandas: 2.3.3
NumPy: 2.3.5


# 2. Importação da Base de Dados

Nesta etapa, será realizada a importação da base de colaboradores para iniciar o diagnóstico e o tratamento dos dados.

In [2]:
caminho_base = Path(
    "../data/raw/People_Analytics_Analise_de_Rotatividade_de_Colaboradores.xlsx"
)

print(caminho_base.exists())

True


In [3]:
# Carrega o arquivo Excel para consultar as planilhas disponíveis
arquivo_excel = pd.ExcelFile(caminho_base)

# Exibe os nomes das planilhas existentes no arquivo
arquivo_excel.sheet_names

['Base Colaboradores', 'Dicionário de Dados']

## 2.1 Carregamento dos Dados

A planilha principal contém os registros dos colaboradores que serão utilizados nas etapas de diagnóstico, tratamento e análise da rotatividade.

In [4]:
# Importa a planilha principal com os registros dos colaboradores
df = pd.read_excel(
    caminho_base,
    sheet_name="Base Colaboradores"
)

# Exibe as cinco primeiras linhas para validar a importação dos dados
df.head()

,colaborador_id,idade,genero,estado_civil,escolaridade,area,cargo,nivel_cargo,data_admissao,tempo_empresa_anos,...,viagens_negocio,satisfacao_trabalho,satisfacao_ambiente,equilibrio_vida_trabalho,avaliacao_desempenho,treinamentos_ultimo_ano,recebeu_promocao_5_anos,rotatividade,data_desligamento,motivo_desligamento
0,COL-0001,38,Masculino,Casado(a),Mestrado,Atendimento,Analista de Atendimento,3,2013-12-13,12,...,Raramente,1.0,1.0,3.0,3.0,3.0,Sim,Sim,2023-07-28 04:45:25.596000,Falta de perspectiva de crescimento
1,COL-0002,61,Feminino,Casado(a),Pós-graduação,Recursos Humanos,Especialista de Pessoas,5,2025-03-07,1,...,Frequentemente,5.0,2.0,2.0,5.0,3.0,Não,Não,Não se aplica,Não se aplica
2,COL-0003,51,Não binário,Casado(a),Graduação,Operações,Coordenador de Operações,4,2022-05-18,4,...,Raramente,4.0,5.0,4.0,3.0,4.0,Não,Não,Não se aplica,Não se aplica
3,COL-0004,34,Feminino,União estável,Tecnólogo,Tecnologia,Analista de Dados,3,2021-05-28,4,...,Raramente,3.0,2.0,4.0,4.0,3.0,Não,Não,Não se aplica,Não se aplica
4,COL-0005,46,Feminino,União estável,Pós-graduação,Financeiro,Controller,3,2015-11-10,11,...,Nunca,3.0,3.0,4.0,3.0,1.0,Sim,Não,Não se aplica,Não se aplica


In [5]:
# Retorna a quantidade de linhas e colunas da base importada
df.shape

(1212, 23)

## 2.2 Diagnóstico Inicial da Base

Nesta etapa, serão verificadas as dimensões, as colunas e os tipos de dados para compreender a estrutura original da base antes do tratamento.

In [6]:
# Apresenta as dimensões da base de forma mais clara
print(f"Quantidade de registros: {df.shape[0]}")
print(f"Quantidade de colunas: {df.shape[1]}")

Quantidade de registros: 1212
Quantidade de colunas: 23


In [7]:
# Exibe os nomes de todas as colunas disponíveis na base
df.columns.tolist()

['colaborador_id',
 'idade',
 'genero',
 'estado_civil',
 'escolaridade',
 'area',
 'cargo',
 'nivel_cargo',
 'data_admissao',
 'tempo_empresa_anos',
 'salario_mensal',
 'distancia_casa_km',
 'horas_extras',
 'viagens_negocio',
 'satisfacao_trabalho',
 'satisfacao_ambiente',
 'equilibrio_vida_trabalho',
 'avaliacao_desempenho',
 'treinamentos_ultimo_ano',
 'recebeu_promocao_5_anos',
 'rotatividade',
 'data_desligamento',
 'motivo_desligamento']

In [8]:
# Apresenta os tipos de dados, valores não nulos e uso de memória
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1212 entries, 0 to 1211
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   colaborador_id            1212 non-null   object        
 1   idade                     1212 non-null   int64         
 2   genero                    1212 non-null   object        
 3   estado_civil              1212 non-null   object        
 4   escolaridade              1206 non-null   object        
 5   area                      1212 non-null   object        
 6   cargo                     1212 non-null   object        
 7   nivel_cargo               1212 non-null   int64         
 8   data_admissao             1212 non-null   datetime64[ns]
 9   tempo_empresa_anos        1212 non-null   int64         
 10  salario_mensal            1206 non-null   float64       
 11  distancia_casa_km         1212 non-null   int64         
 12  horas_extras        

## 2.3 Identificação de Valores Ausentes

A presença de valores ausentes pode comprometer cálculos, visualizações e análises estatísticas. Nesta etapa, será identificada a quantidade e a proporção de ausências em cada variável.

In [9]:
# Calcula a quantidade de valores ausentes em cada coluna
quantidade_ausentes = df.isna().sum()

# Calcula o percentual de valores ausentes em relação ao total de registros
percentual_ausentes = (df.isna().mean() * 100).round(2)

# Cria uma tabela resumida com a quantidade e o percentual de ausências
resumo_ausentes = pd.DataFrame({
    "quantidade_ausentes": quantidade_ausentes,
    "percentual_ausentes": percentual_ausentes
})

# Mantém somente as colunas que possuem pelo menos um valor ausente
resumo_ausentes = resumo_ausentes[
    resumo_ausentes["quantidade_ausentes"] > 0
].sort_values(
    by="quantidade_ausentes",
    ascending=False
)

# Exibe o resumo de valores ausentes
resumo_ausentes

,quantidade_ausentes,percentual_ausentes
escolaridade,6,0.50
salario_mensal,6,0.50
viagens_negocio,6,0.50
satisfacao_trabalho,5,0.41
satisfacao_ambiente,5,0.41
equilibrio_vida_trabalho,5,0.41
avaliacao_desempenho,5,0.41
treinamentos_ultimo_ano,5,0.41


In [10]:
# Calcula o total de valores ausentes existentes em toda a base
total_ausentes = df.isna().sum().sum()

# Apresenta o total encontrado para validar a qualidade da base bruta
print(f"Total de valores ausentes: {total_ausentes}")

Total de valores ausentes: 43


## 2.4 Identificação de Registros Duplicados

Registros duplicados podem distorcer indicadores como quantidade de colaboradores, taxa de rotatividade e médias salariais. Nesta etapa, serão identificadas as duplicidades exatas antes do tratamento.

In [11]:
# Identifica a quantidade de registros completamente duplicados na base
quantidade_duplicados = df.duplicated().sum()

# Apresenta o total de duplicidades encontradas
print(f"Quantidade de registros duplicados: {quantidade_duplicados}")

Quantidade de registros duplicados: 12


In [12]:
# Seleciona somente as ocorrências duplicadas, desconsiderando a primeira ocorrência
registros_duplicados = df[df.duplicated()].copy()

# Exibe os registros duplicados para conferência
registros_duplicados

,colaborador_id,idade,genero,estado_civil,escolaridade,area,cargo,nivel_cargo,data_admissao,tempo_empresa_anos,...,viagens_negocio,satisfacao_trabalho,satisfacao_ambiente,equilibrio_vida_trabalho,avaliacao_desempenho,treinamentos_ultimo_ano,recebeu_promocao_5_anos,rotatividade,data_desligamento,motivo_desligamento
1200,COL-0001,38,Masculino,Casado(a),Mestrado,Atendimento,Analista de Atendimento,3,2013-12-13,12,...,Raramente,1.0,1.0,3.0,3.0,3.0,Sim,Sim,2023-07-28 04:45:25.596000,Falta de perspectiva de crescimento
1201,COL-0002,61,Feminino,Casado(a),Pós-graduação,Recursos Humanos,Especialista de Pessoas,5,2025-03-07,1,...,Frequentemente,5.0,2.0,2.0,5.0,3.0,Não,Não,Não se aplica,Não se aplica
1202,COL-0003,51,Não binário,Casado(a),Graduação,Operações,Coordenador de Operações,4,2022-05-18,4,...,Raramente,4.0,5.0,4.0,3.0,4.0,Não,Não,Não se aplica,Não se aplica
1203,COL-0004,34,Feminino,União estável,Tecnólogo,Tecnologia,Analista de Dados,3,2021-05-28,4,...,Raramente,3.0,2.0,4.0,4.0,3.0,Não,Não,Não se aplica,Não se aplica
1204,COL-0005,46,Feminino,União estável,Pós-graduação,Financeiro,Controller,3,2015-11-10,11,...,Nunca,3.0,3.0,4.0,3.0,1.0,Sim,Não,Não se aplica,Não se aplica
1205,COL-0006,50,Masculino,Divorciado(a),Graduação,Operações,Coordenador de Operações,3,2019-04-09,7,...,Raramente,1.0,5.0,5.0,3.0,2.0,Sim,Não,Não se aplica,Não se aplica
1206,COL-0007,35,Não binário,União estável,Graduação,Atendimento,Analista de Atendimento,3,2021-06-04,4,...,Frequentemente,1.0,2.0,2.0,3.0,1.0,Sim,Sim,2023-03-23 19:04:09.287000,Desempenho abaixo do esperado
1207,COL-0008,36,Feminino,União estável,Tecnólogo,Atendimento,Assistente de Atendimento,2,2019-08-21,6,...,Raramente,4.0,4.0,4.0,5.0,5.0,Não,Não,Não se aplica,Não se aplica
1208,COL-0009,40,Feminino,Divorciado(a),Graduação,Comercial,Executivo de Contas,2,2023-12-10,3,...,Raramente,5.0,4.0,4.0,3.0,6.0,Não,Não,Não se aplica,Não se aplica
1209,COL-0010,60,Feminino,Casado(a),Ensino médio,Marketing,Coordenador de Marketing,5,2013-06-15,12,...,Frequentemente,1.0,3.0,4.0,3.0,0.0,Sim,Não,Não se aplica,Não se aplica


In [13]:
# Exibe os identificadores dos colaboradores que possuem registros duplicados
registros_duplicados["colaborador_id"].tolist()

['COL-0001',
 'COL-0002',
 'COL-0003',
 'COL-0004',
 'COL-0005',
 'COL-0006',
 'COL-0007',
 'COL-0008',
 'COL-0009',
 'COL-0010',
 'COL-0011',
 'COL-0012']

# 3. Tratamento e Limpeza dos Dados

Nesta etapa, será criada uma cópia da base original para realizar os tratamentos necessários. Essa prática preserva os dados brutos e garante rastreabilidade durante o processo de limpeza.

## 3.1 Remoção de Registros Duplicados

As duplicidades exatas serão removidas, mantendo apenas a primeira ocorrência de cada registro.

In [14]:
# Cria uma cópia da base original para preservar os dados brutos
df_bruto = df.copy()

# Cria a base que será utilizada nas etapas de tratamento e análise
df_tratado = df.copy()

In [15]:
# Remove os registros completamente duplicados e reorganiza o índice
df_tratado = (
    df_tratado
    .drop_duplicates()
    .reset_index(drop=True)
)

# Calcula quantos registros foram removidos durante o tratamento
registros_removidos = len(df_bruto) - len(df_tratado)

# Apresenta o resultado da remoção das duplicidades
print(f"Registros antes do tratamento: {len(df_bruto)}")
print(f"Registros após o tratamento: {len(df_tratado)}")
print(f"Registros duplicados removidos: {registros_removidos}")

Registros antes do tratamento: 1212
Registros após o tratamento: 1200
Registros duplicados removidos: 12


In [16]:
# Confirma que não existem mais registros duplicados na base tratada
print(f"Duplicidades restantes: {df_tratado.duplicated().sum()}")

# Confirma que os 43 valores ausentes permanecem para o próximo tratamento
print(f"Valores ausentes restantes: {df_tratado.isna().sum().sum()}")

Duplicidades restantes: 0
Valores ausentes restantes: 43


## 3.2 Análise dos Valores Ausentes

Antes da aplicação de qualquer tratamento, serão analisados os registros e as variáveis afetadas. O objetivo é definir estratégias adequadas de preenchimento de acordo com o tipo e o significado de cada dado.

In [17]:
# Identifica as colunas que ainda possuem valores ausentes na base tratada
colunas_com_ausentes = df_tratado.columns[
    df_tratado.isna().any()
].tolist()

# Exibe a relação de colunas que precisam de tratamento
colunas_com_ausentes

['escolaridade',
 'salario_mensal',
 'viagens_negocio',
 'satisfacao_trabalho',
 'satisfacao_ambiente',
 'equilibrio_vida_trabalho',
 'avaliacao_desempenho',
 'treinamentos_ultimo_ano']

In [18]:
# Seleciona os registros que apresentam pelo menos um valor ausente
linhas_com_ausentes = df_tratado[
    df_tratado[colunas_com_ausentes].isna().any(axis=1)
].copy()

# Apresenta a quantidade de registros afetados
print(f"Registros com pelo menos um valor ausente: {len(linhas_com_ausentes)}")

Registros com pelo menos um valor ausente: 43


In [19]:
# Exibe uma amostra dos colaboradores e das variáveis que possuem valores ausentes
linhas_com_ausentes[
    ["colaborador_id"] + colunas_com_ausentes
].head(15)

,colaborador_id,escolaridade,salario_mensal,viagens_negocio,satisfacao_trabalho,satisfacao_ambiente,equilibrio_vida_trabalho,avaliacao_desempenho,treinamentos_ultimo_ano
100,COL-0101,NaN,9200.0,Raramente,2.0,5.0,1.0,5.0,2.0
101,COL-0102,Graduação,NaN,Nunca,4.0,2.0,2.0,4.0,2.0
102,COL-0103,Pós-graduação,8800.0,NaN,4.0,5.0,5.0,2.0,2.0
103,COL-0104,Pós-graduação,9300.0,Raramente,NaN,1.0,1.0,3.0,6.0
104,COL-0105,Graduação,7100.0,Raramente,3.0,NaN,2.0,3.0,1.0
105,COL-0106,Pós-graduação,9900.0,Raramente,5.0,1.0,NaN,2.0,1.0
106,COL-0107,Mestrado,8450.0,Frequentemente,4.0,5.0,3.0,NaN,0.0
107,COL-0108,Graduação,8250.0,Raramente,3.0,2.0,5.0,3.0,NaN
108,COL-0109,NaN,7650.0,Raramente,3.0,2.0,3.0,2.0,3.0
109,COL-0110,Graduação,NaN,Nunca,5.0,1.0,4.0,5.0,3.0


In [20]:
# Classifica as variáveis com ausências de acordo com o tipo de dado
colunas_categoricas_ausentes = df_tratado[
    colunas_com_ausentes
].select_dtypes(include="object").columns.tolist()

# Identifica as variáveis numéricas que possuem valores ausentes
colunas_numericas_ausentes = df_tratado[
    colunas_com_ausentes
].select_dtypes(include="number").columns.tolist()

# Apresenta a classificação das variáveis
print("Variáveis categóricas:", colunas_categoricas_ausentes)
print("Variáveis numéricas:", colunas_numericas_ausentes)

Variáveis categóricas: ['escolaridade', 'viagens_negocio']
Variáveis numéricas: ['salario_mensal', 'satisfacao_trabalho', 'satisfacao_ambiente', 'equilibrio_vida_trabalho', 'avaliacao_desempenho', 'treinamentos_ultimo_ano']


## 3.3 Tratamento dos Valores Ausentes

As variáveis categóricas serão preenchidas com a categoria “Não informado”, preservando a transparência sobre a ausência original.

Para o salário mensal, será utilizada a mediana da combinação entre área e nível do cargo. Nas variáveis de satisfação, desempenho e treinamento, será aplicada a mediana da área. A mediana foi escolhida por ser menos sensível a valores extremos.

In [21]:
# Registra a quantidade de valores ausentes antes do tratamento
ausentes_antes_tratamento = df_tratado.isna().sum().sum()

# Apresenta o total que será tratado
print(f"Valores ausentes antes do tratamento: {ausentes_antes_tratamento}")

Valores ausentes antes do tratamento: 43


In [22]:
# Preenche as variáveis categóricas ausentes sem atribuir uma informação presumida
df_tratado[colunas_categoricas_ausentes] = (
    df_tratado[colunas_categoricas_ausentes]
    .fillna("Não informado")
)

In [23]:
# Calcula a mediana salarial considerando a área e o nível do cargo
mediana_salario_grupo = (
    df_tratado
    .groupby(["area", "nivel_cargo"])["salario_mensal"]
    .transform("median")
)

# Preenche salários ausentes com a mediana do grupo e utiliza a mediana geral como alternativa
df_tratado["salario_mensal"] = (
    df_tratado["salario_mensal"]
    .fillna(mediana_salario_grupo)
    .fillna(df_tratado["salario_mensal"].median())
)

In [24]:
# Define as variáveis numéricas discretas que precisam de tratamento
colunas_numericas_discretas = [
    "satisfacao_trabalho",
    "satisfacao_ambiente",
    "equilibrio_vida_trabalho",
    "avaliacao_desempenho",
    "treinamentos_ultimo_ano"
]

# Preenche cada variável com a mediana da respectiva área organizacional
for coluna in colunas_numericas_discretas:
    mediana_por_area = (
        df_tratado
        .groupby("area")[coluna]
        .transform("median")
    )

    df_tratado[coluna] = (
        df_tratado[coluna]
        .fillna(mediana_por_area)
        .fillna(df_tratado[coluna].median())
        .round()
        .astype(int)
    )

In [25]:
# Calcula a quantidade de valores ausentes após o tratamento
ausentes_depois_tratamento = df_tratado.isna().sum().sum()

# Apresenta o resultado final do tratamento
print(f"Valores ausentes antes: {ausentes_antes_tratamento}")
print(f"Valores ausentes depois: {ausentes_depois_tratamento}")
print(
    f"Valores tratados: "
    f"{ausentes_antes_tratamento - ausentes_depois_tratamento}"
)

Valores ausentes antes: 43
Valores ausentes depois: 0
Valores tratados: 43


## 3.4 Padronização dos Tipos de Dados

A coluna de desligamento contém datas para ex-colaboradores e o texto “Não se aplica” para colaboradores ativos. Para permitir análises temporais, o campo será convertido para o formato de data.

A ausência de uma data de desligamento para colaboradores ativos é uma condição esperada e não representa um problema de qualidade.

In [26]:
# Converte a coluna de desligamento para data.
# O texto "Não se aplica" (colaboradores ativos) é convertido automaticamente em NaT.
# O horário é descartado, pois a análise utiliza apenas a data do desligamento.
df_tratado["data_desligamento"] = pd.to_datetime(
    df_tratado["data_desligamento"],
    errors="coerce"
).dt.normalize()

In [27]:
# Confirma os tipos de dados das colunas de admissão e desligamento
df_tratado[
    ["data_admissao", "data_desligamento"]
].dtypes

data_admissao        datetime64[ns]
data_desligamento    datetime64[ns]
dtype: object

In [28]:
# Identifica colaboradores ativos que possuem uma data de desligamento indevida
ativos_com_data_desligamento = df_tratado[
    (df_tratado["rotatividade"] == "Não")
    & (df_tratado["data_desligamento"].notna())
]

# Identifica colaboradores desligados que não possuem uma data de desligamento
desligados_sem_data = df_tratado[
    (df_tratado["rotatividade"] == "Sim")
    & (df_tratado["data_desligamento"].isna())
]

# Apresenta possíveis inconsistências entre rotatividade e data de desligamento
print(
    "Ativos com data de desligamento:",
    len(ativos_com_data_desligamento)
)

print(
    "Desligados sem data de desligamento:",
    len(desligados_sem_data)
)

Ativos com data de desligamento: 0
Desligados sem data de desligamento: 0


## 3.5 Recálculo do Tempo de Empresa

O dicionário de dados descreve `tempo_empresa_anos` como um valor **aproximado**. Ao comparar o campo com as datas, foram identificadas inconsistências: para colaboradores desligados, o tempo era contado até a data atual e não até a data de saída (por exemplo, COL-0001 foi admitido em 2013, desligado em 2023 e aparecia com 12 anos de empresa).

Como a análise de rotatividade por tempo de empresa depende desse campo, ele será recalculado a partir das datas:

- **Desligados:** da admissão até a data de desligamento;
- **Ativos:** da admissão até a data de referência da base (data mais recente registrada).

O valor original é preservado na coluna `tempo_empresa_anos_informado` para rastreabilidade.

In [29]:
# Define a data de referência como a data mais recente registrada na base
data_referencia = max(
    df_tratado["data_admissao"].max(),
    df_tratado["data_desligamento"].max()
)

print(f"Data de referência da base: {data_referencia:%d/%m/%Y}")

Data de referência da base: 09/09/2026


In [30]:
# Preserva o valor original para rastreabilidade
df_tratado["tempo_empresa_anos_informado"] = df_tratado["tempo_empresa_anos"]

# Para desligados, usa a data de saída; para ativos, a data de referência
data_fim = df_tratado["data_desligamento"].fillna(data_referencia)

# Recalcula o tempo de empresa em anos completos
df_tratado["tempo_empresa_anos"] = (
    (data_fim - df_tratado["data_admissao"]).dt.days // 365.25
).astype(int)

# Mede o tamanho da correção por situação do colaborador
diferenca = (
    df_tratado["tempo_empresa_anos_informado"]
    - df_tratado["tempo_empresa_anos"]
)

(
    df_tratado
    .assign(diferenca_anos=diferenca)
    .groupby("rotatividade")["diferenca_anos"]
    .agg(
        registros="size",
        registros_alterados=lambda s: s.ne(0).sum(),
        diferenca_media="mean"
    )
    .round(2)
)

,registros,registros_alterados,diferenca_media
rotatividade,,,
Não,877,469,-0.31
Sim,323,241,1.48


## 3.6 Validação da Qualidade dos Dados

Após os tratamentos, serão aplicados testes de qualidade para verificar unicidade, completude, consistência e validade dos dados. O objetivo é garantir que a base esteja adequada para as análises posteriores.

In [31]:
# Define as colunas de satisfação que devem apresentar valores entre 1 e 5
colunas_satisfacao = [
    "satisfacao_trabalho",
    "satisfacao_ambiente",
    "equilibrio_vida_trabalho",
    "avaliacao_desempenho"
]

# Verifica se todas as escalas de satisfação possuem valores válidos
escalas_validas = (
    df_tratado[colunas_satisfacao]
    .apply(lambda coluna: coluna.between(1, 5))
    .all()
    .all()
)

In [32]:
# Cria um conjunto de testes para avaliar a qualidade da base tratada
validacoes = {
    "Total de registros igual a 1.200":
        len(df_tratado) == 1200,

    "Identificadores únicos":
        df_tratado["colaborador_id"].nunique() == len(df_tratado),

    "Ausência de registros duplicados":
        df_tratado.duplicated().sum() == 0,

    "Campos obrigatórios sem valores ausentes":
        df_tratado.drop(columns=["data_desligamento"])
        .isna()
        .sum()
        .sum() == 0,

    "Idades entre 18 e 70 anos":
        df_tratado["idade"].between(18, 70).all(),

    "Níveis de cargo entre 1 e 5":
        df_tratado["nivel_cargo"].between(1, 5).all(),

    "Salários maiores que zero":
        df_tratado["salario_mensal"].gt(0).all(),

    "Escalas de avaliação entre 1 e 5":
        escalas_validas,

    "Treinamentos com valores não negativos":
        df_tratado["treinamentos_ultimo_ano"].ge(0).all(),

    "Rotatividade padronizada em Sim ou Não":
        df_tratado["rotatividade"].isin(["Sim", "Não"]).all(),

    "Datas de desligamento posteriores à admissão":
        (
            df_tratado["data_desligamento"].isna()
            | (
                df_tratado["data_desligamento"]
                >= df_tratado["data_admissao"]
            )
        ).all(),

    "Ativos sem data de desligamento":
        len(ativos_com_data_desligamento) == 0,

    "Desligados com data de desligamento":
        len(desligados_sem_data) == 0,

    "Tempo de empresa coerente com as datas":
        df_tratado["tempo_empresa_anos"].between(0, 60).all()
        and (
            df_tratado["tempo_empresa_anos"]
            == (
                (
                    df_tratado["data_desligamento"].fillna(data_referencia)
                    - df_tratado["data_admissao"]
                ).dt.days // 365.25
            )
        ).all()
}

In [33]:
# Converte os testes de qualidade em uma tabela de validação
relatorio_validacao = pd.DataFrame(
    validacoes.items(),
    columns=["teste", "resultado"]
)

# Converte os resultados booleanos em uma classificação de fácil leitura
relatorio_validacao["status"] = np.where(
    relatorio_validacao["resultado"],
    "Aprovado",
    "Revisar"
)

# Exibe o relatório final de qualidade dos dados
relatorio_validacao

,teste,resultado,status
0,Total de registros igual a 1.200,True,Aprovado
1,Identificadores únicos,True,Aprovado
2,Ausência de registros duplicados,True,Aprovado
3,Campos obrigatórios sem valores ausentes,True,Aprovado
4,Idades entre 18 e 70 anos,True,Aprovado
5,Níveis de cargo entre 1 e 5,True,Aprovado
6,Salários maiores que zero,True,Aprovado
7,Escalas de avaliação entre 1 e 5,True,Aprovado
8,Treinamentos com valores não negativos,True,Aprovado
9,Rotatividade padronizada em Sim ou Não,True,Aprovado


In [34]:
# Apresenta um resumo do resultado geral da validação
if relatorio_validacao["resultado"].all():
    print("Todos os testes de qualidade foram aprovados.")
else:
    print("Existem testes que precisam ser revisados.")

Todos os testes de qualidade foram aprovados.


## 3.7 Exportação da Base Tratada

Após a conclusão dos processos de limpeza e validação, a base tratada será exportada nos formatos Excel e CSV.

O arquivo original será preservado para garantir rastreabilidade, enquanto os novos arquivos serão utilizados nas análises em Python, SQL e Power BI.

In [35]:
# Define os caminhos onde os arquivos tratados serão armazenados
caminho_excel_tratado = Path(
    "../data/processed/People_Analytics_Base_Tratada.xlsx"
)

caminho_csv_tratado = Path(
    "../data/processed/People_Analytics_Base_Tratada.csv"
)

In [36]:
# Exporta a base tratada e o relatório de validação para um arquivo Excel
with pd.ExcelWriter(
    caminho_excel_tratado,
    engine="openpyxl"
) as writer:

    df_tratado.to_excel(
        writer,
        sheet_name="Base Tratada",
        index=False
    )

    relatorio_validacao.to_excel(
        writer,
        sheet_name="Validação de Qualidade",
        index=False
    )

In [37]:
# Exporta a base tratada para CSV com codificação compatível com caracteres em português
df_tratado.to_csv(
    caminho_csv_tratado,
    index=False,
    encoding="utf-8-sig"
)

In [38]:
# Confirma se os arquivos tratados foram criados corretamente
print(
    "Arquivo Excel criado:",
    caminho_excel_tratado.exists()
)

print(
    "Arquivo CSV criado:",
    caminho_csv_tratado.exists()
)

Arquivo Excel criado: True
Arquivo CSV criado: True
